In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [74]:
df = pd.read_csv("data__movies.csv", sep=",")

In [75]:
df= df.drop(columns=["homepage", "status", "tagline", "title","original_title", "overview", "id"])

In [76]:
df['production_companies'] = df['production_companies'].str.split(',').str[0]
df = df.map(lambda x: x.lower() if isinstance(x, str) else x)

In [77]:
df["genres"] = df["genres"].fillna("desconocido")
df["production_companies"] = df["production_companies"].fillna("otros")
df = df.dropna(subset=["release_date", "runtime"])

In [78]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df = df.dropna(subset=["release_date"])
df["anio"] = df["release_date"].dt.year.astype(int)
df["mes"] = df["release_date"].dt.month.astype(int)
df = df.drop(columns=["release_date"])

In [79]:
df = df[(df["budget"] > 0) & (df["revenue"] > 0)
              & (df["runtime"] > 0) & (df["vote_count"] > 0)]


In [80]:
generos_exp = df["genres"].str.split(",").apply(
    lambda l: [g.strip() for g in l] if isinstance(l, list) else [])

In [81]:
top_generos = pd.Series([g for sub in generos_exp for g in sub]) \
                .value_counts().head(10).index.tolist()

In [82]:
def _limpiar(s):
    out = s
    for ch in [" ", ".", ",", "-", "(", ")", "/", "'", "&"]:
        out = out.replace(ch, "_")
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")

In [83]:
for g in top_generos:
    df[f"gen_{_limpiar(g)}"] = generos_exp.apply(lambda lst: int(g in lst))
df["n_generos"] = generos_exp.apply(len)

In [84]:
prods_exp = df["production_companies"].str.split(",").apply(
    lambda l: [p.strip() for p in l] if isinstance(l, list) else [])
top_prods = pd.Series([p for sub in prods_exp for p in sub]) \
              .value_counts().head(10).index.tolist()
for p in top_prods:
    df[f"prod_{_limpiar(p)}"] = prods_exp.apply(lambda lst: int(p in lst))

top_lang = df["original_language"].value_counts().head(5).index
df["original_language"] = df["original_language"].where(
    df["original_language"].isin(top_lang), "other")

In [85]:
df["budget"]     = np.log(df["budget"])
df["revenue"] = np.log(df["revenue"])
df["popularity"] = np.log1p(df["popularity"])
df["vote_count"] = np.log1p(df["vote_count"])

In [86]:
q_low  = df["revenue"].quantile(0.010)
q_high = df["revenue"].quantile(0.990)
df = df[(df["revenue"] >= q_low) & (df["revenue"] <= q_high)].copy()

for col in ["budget", "popularity", "vote_count",
            "vote_average", "runtime", "anio"]:
    df[col] = df[col] - df[col].mean()

In [87]:
df = df.reset_index(drop=True)

In [88]:
print(f"df_v2 final: {len(df):,} filas, {df.shape[1]} columnas")
print(f"Columnas creadas para géneros: {[c for c in df.columns if c.startswith('gen_')]}")
print(f"Columnas creadas para productoras: {[c for c in df.columns if c.startswith('prod_')]}")
df.head()

df_v2 final: 3,161 filas, 32 columnas
Columnas creadas para géneros: ['gen_drama', 'gen_comedy', 'gen_thriller', 'gen_action', 'gen_adventure', 'gen_romance', 'gen_crime', 'gen_science_fiction', 'gen_family', 'gen_fantasy']
Columnas creadas para productoras: ['prod_paramount_pictures', 'prod_universal_pictures', 'prod_columbia_pictures', 'prod_twentieth_century_fox_film_corporation', 'prod_new_line_cinema', 'prod_walt_disney_pictures', 'prod_village_roadshow_pictures', 'prod_united_artists', 'prod_miramax_films', 'prod_columbia_pictures_corporation']


,budget,genres,original_language,popularity,production_companies,revenue,runtime,vote_average,vote_count,anio,...,prod_paramount_pictures,prod_universal_pictures,prod_columbia_pictures,prod_twentieth_century_fox_film_corporation,prod_new_line_cinema,prod_walt_disney_pictures,prod_village_roadshow_pictures,prod_united_artists,prod_miramax_films,prod_columbia_pictures_corporation
0,2.482169,"action, adventure, crime",en,1.690763,columbia pictures,20.596199,37.473268,-0.00658,2.371210,13.401455,...,0,0,1,0,0,0,0,0,0,0
1,2.541593,"action, adventure, science fiction",en,0.810188,walt disney pictures,19.464974,21.473268,-0.20658,1.628265,10.401455,...,0,0,0,0,0,1,0,0,0,0
2,2.533870,"fantasy, action, adventure",en,1.764754,columbia pictures,20.607711,28.473268,-0.40658,2.149017,5.401455,...,0,0,1,0,0,0,0,0,0,0
3,2.541593,"animation, family",en,0.910791,walt disney pictures,20.198671,-10.526732,1.09342,2.077766,8.401455,...,0,0,0,0,0,1,0,0,0,0
4,2.502372,"action, adventure, fantasy",en,2.060059,dc comics,20.587744,40.473268,-0.60658,2.821117,14.401455,...,0,0,0,0,0,0,0,0,0,0


In [89]:
formula_final = '''
revenue ~ vote_count + budget + anio
+ gen_family + gen_science_fiction + gen_crime + gen_fantasy + gen_romance + gen_drama
+ vote_average
+ prod_new_line_cinema + prod_twentieth_century_fox_film_corporation
+ prod_paramount_pictures + prod_universal_pictures + prod_columbia_pictures
+ prod_miramax_films + prod_village_roadshow_pictures
+ runtime
+ budget:gen_crime
+ budget:gen_science_fiction
+ budget:gen_romance
+ budget:gen_fantasy
+ budget:gen_thriller
+ budget:vote_average
+ budget:runtime
+ budget:prod_twentieth_century_fox_film_corporation
+ budget:prod_new_line_cinema
+ vote_average:popularity
+ popularity:vote_count
'''

modelo_final = smf.ols(formula_final, data=df).fit()
print(modelo_final.summary())

                            OLS Regression Results                            
Dep. Variable:                revenue   R-squared:                       0.671
Model:                            OLS   Adj. R-squared:                  0.668
Method:                 Least Squares   F-statistic:                     220.1
Date:                Sun, 26 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:48:01   Log-Likelihood:                -4409.3
No. Observations:                3161   AIC:                             8879.
Df Residuals:                    3131   BIC:                             9060.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [90]:
# 1. Obtenemos las métricas de influencia y los residuos estudentizados del modelo actual
influencia = modelo_final.get_influence()
residuos_estudentizados = influencia.resid_studentized_internal

# 2. Filtramos los valores atípicos severos (errores estándar absolutos mayores a 2.0 o 2.5)
# Un umbral de 2.0 es más estricto y garantiza máxima normalidad.
df_limpio = df[abs(residuos_estudentizados) <= 2].copy()

# 3. Volvemos a ajustar el modelo con el dataframe limpio
modelo_corregido = smf.ols(formula_final, data=df_limpio).fit()

# 4. Mostramos los resultados
print(f"Observaciones eliminadas: {len(df) - len(df_limpio)}")
print(modelo_corregido.summary())

Observaciones eliminadas: 161
                            OLS Regression Results                            
Dep. Variable:                revenue   R-squared:                       0.768
Model:                            OLS   Adj. R-squared:                  0.766
Method:                 Least Squares   F-statistic:                     338.6
Date:                Sun, 26 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:48:01   Log-Likelihood:                -3220.2
No. Observations:                3000   AIC:                             6500.
Df Residuals:                    2970   BIC:                             6681.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------

In [91]:
df_limpio

,budget,genres,original_language,popularity,production_companies,revenue,runtime,vote_average,vote_count,anio,...,prod_paramount_pictures,prod_universal_pictures,prod_columbia_pictures,prod_twentieth_century_fox_film_corporation,prod_new_line_cinema,prod_walt_disney_pictures,prod_village_roadshow_pictures,prod_united_artists,prod_miramax_films,prod_columbia_pictures_corporation
0,2.482169,"action, adventure, crime",en,1.690763,columbia pictures,20.596199,37.473268,-0.00658,2.371210,13.401455,...,0,0,1,0,0,0,0,0,0,0
1,2.541593,"action, adventure, science fiction",en,0.810188,walt disney pictures,19.464974,21.473268,-0.20658,1.628265,10.401455,...,0,0,0,0,0,1,0,0,0,0
2,2.533870,"fantasy, action, adventure",en,1.764754,columbia pictures,20.607711,28.473268,-0.40658,2.149017,5.401455,...,0,0,1,0,0,0,0,0,0,0
3,2.541593,"animation, family",en,0.910791,walt disney pictures,20.198671,-10.526732,1.09342,2.077766,8.401455,...,0,0,0,0,0,1,0,0,0,0
4,2.502372,"action, adventure, fantasy",en,2.060059,dc comics,20.587744,40.473268,-0.60658,2.821117,14.401455,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3154,-5.752457,"romance, comedy, drama",en,-1.776877,tiny ponies,12.939637,-11.526732,-0.70658,-1.938918,8.401455,...,0,0,0,0,0,0,0,0,0,0
3156,-6.631008,comedy,en,0.037631,miramax films,14.963272,-18.526732,1.09342,0.594779,-7.598545,...,0,0,0,0,0,0,0,0,1,0
3158,-6.931112,"crime, horror, mystery, thriller",ja,-2.802214,daiei studios,11.502875,0.473268,1.09342,-1.874379,-4.598545,...,0,0,0,0,0,0,0,0,0,0
3159,-7.980934,"science fiction, drama, thriller",en,0.195952,thinkfilm,12.959280,-33.526732,0.59342,0.457461,2.401455,...,0,0,0,0,0,0,0,0,0,0
